**Limpieza e Imputación de datos**





In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [13]:
import os
import pandas as pd

ruta_archivo = '/Dataset analitica'

# Lectura del dataset
df_raw = pd.read_excel(ruta_archivo, sheet_name="Datos semanales")
df = df_raw.copy()

df['nodo'] = df['nodo'].astype(str)

print("=== DIAGNÓSTICO INICIAL ===")
print(f"Dimensiones del dataset: {df.shape[0]} registros y {df.shape[1]} columnas.")
print(f"Nodos evaluados: {sorted(df['nodo'].unique(), key=int)}")
print(f"Rango temporal: {df['inicio_semana'].min().strftime('%Y-%m-%d')} a {df['inicio_semana'].max().strftime('%Y-%m-%d')}")
print(f"Total de celdas nulas iniciales: {df.isna().sum().sum()}")

=== DIAGNÓSTICO INICIAL ===
Dimensiones del dataset: 2873 registros y 49 columnas.
Nodos evaluados: ['1', '2', '3', '4', '5', '6', '7', '8', '10', '11', '12', '13', '14', '15', '16', '17', '18']
Rango temporal: 2023-06-26 a 2026-09-14
Total de celdas nulas iniciales: 509


**Valores atipicos e incoherencias**

In [14]:
bitacora = []

def registrar_evento(fase, variable, criterio, accion, registros_afectados):
    """Función auxiliar para registrar decisiones en la bitácora."""
    bitacora.append({
        'Fase': fase,
        'Variable(s)': variable,
        'Criterio / Regla de Negocio': criterio,
        'Acción Aplicada': accion,
        'Registros Afectados': registros_afectados
    })

# a) Temperaturas físicamente imposibles (> 50 °C)
filas_temp_invalida = df['temperatura_max_c'] > 50
n_temp = filas_temp_invalida.sum()

cols_temp = ['temperatura_media_c', 'temperatura_min_c', 'temperatura_max_c', 'amplitud_termica_c']
df.loc[filas_temp_invalida, cols_temp] = np.nan
df.loc[df['temperatura_2sem_previas_c'] > 50, 'temperatura_2sem_previas_c'] = np.nan

registrar_evento(
    'Limpieza Atípicos',
    'Temperatura (media, min, max, amplitud)',
    'Valores de temperatura máxima > 50 °C (físicamente imposibles)',
    'Reemplazo por NaN para posterior imputación temporal',
    n_temp
)

# b) Humedad relativa igual a 0% (fallo de sensor)
filas_hum_cero = df['humedad_media_pct'] == 0
n_hum = filas_hum_cero.sum()

cols_hum = ['humedad_min_pct', 'humedad_media_pct', 'humedad_max_pct']
df.loc[filas_hum_cero, cols_hum] = np.nan
df.loc[df['humedad_2sem_previas_pct'] == 0, 'humedad_2sem_previas_pct'] = np.nan

registrar_evento(
    'Limpieza Atípicos',
    'Humedad Relativa',
    'Humedad media o previa igual a 0 % (fallo de sensor)',
    'Reemplazo por NaN para posterior imputación',
    n_hum
)

# c) Radiación media igual a 0
filas_rad_cero = df['radiacion_media'] == 0
n_rad = filas_rad_cero.sum()

cols_rad = ['radiacion_media', 'radiacion_max', 'uv_media', 'uv_max']
df.loc[filas_rad_cero, cols_rad] = np.nan

registrar_evento(
    'Limpieza Atípicos',
    'Radiación y Radiación UV',
    'Radiación media semanal igual a 0',
    'Reemplazo por NaN para posterior imputación',
    n_rad
)

# d) Incoherencias lógicas (Mínima > Media)
filas_inc_hum = df['humedad_min_pct'] > df['humedad_media_pct']
df.loc[filas_inc_hum, cols_hum] = np.nan

registrar_evento(
    'Limpieza Incoherencias',
    'Humedad Relativa',
    'Incoherencia lógica: Humedad Mínima > Humedad Media',
    'Reemplazo del bloque por NaN',
    filas_inc_hum.sum()
)

print("Depuración de inconsistencias completada.")

Depuración de inconsistencias completada.


**Imputacion por nodo**

In [15]:
# Ordenar cronológicamente
df = df.sort_values(by=['nodo', 'inicio_semana']).reset_index(drop=True)

# Identificar columnas
cols_excluir = ['nodo', 'semana', 'año', 'inicio_semana', 'fin_semana']
cols_imputar = [col for col in df.select_dtypes(include=[np.number]).columns if col not in cols_excluir]

# Contar nulos acumulados
total_nulos_pre = df[cols_imputar].isna().sum().sum()

# Interpolación lineal intra-nodo para huecos cortos
for col in cols_imputar:
    df[col] = df.groupby('nodo')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))

# Mediana espacio-temporal (Año-Mes) para huecos continuos
df['mes'] = df['inicio_semana'].dt.month
for col in cols_imputar:
    if df[col].isna().sum() > 0:
        df[col] = df.groupby(['año', 'mes'])[col].transform(lambda x: x.fillna(x.median()))

# Mediana histórica del nodo si persisten bordes sin imputar
for col in cols_imputar:
    if df[col].isna().sum() > 0:
        df[col] = df.groupby('nodo')[col].transform(lambda x: x.fillna(x.median()))

# Imputación controlada de la variable respuesta 'huevos_aedes' por mediana del nodo
df['huevos_aedes'] = df.groupby('nodo')['huevos_aedes'].transform(lambda x: x.fillna(x.median()))

registrar_evento(
    'Imputación de Datos',
    'Variables Climáticas y Entomológicas',
    'Presencia de vacíos (especialmente en Nodos 15 y 16 por fallos de red)',
    'Interpolación lineal intra-nodo secuencial + Mediana espacio-temporal (Año-Mes)',
    total_nulos_pre
)

print(f"Imputación finalizada. Nulos residuales en el dataset: {df[cols_imputar].isna().sum().sum()}")

Imputación finalizada. Nulos residuales en el dataset: 0


**Bitacora**

In [16]:
df_bitacora = pd.DataFrame(bitacora)

pd.set_option('display.max_colwidth', None)
display(df_bitacora)

,Fase,Variable(s),Criterio / Regla de Negocio,Acción Aplicada,Registros Afectados
0,Limpieza Atípicos,"Temperatura (media, min, max, amplitud)",Valores de temperatura máxima > 50 °C (físicamente imposibles),Reemplazo por NaN para posterior imputación temporal,6
1,Limpieza Atípicos,Humedad Relativa,Humedad media o previa igual a 0 % (fallo de sensor),Reemplazo por NaN para posterior imputación,5
2,Limpieza Atípicos,Radiación y Radiación UV,Radiación media semanal igual a 0,Reemplazo por NaN para posterior imputación,2
3,Limpieza Incoherencias,Humedad Relativa,Incoherencia lógica: Humedad Mínima > Humedad Media,Reemplazo del bloque por NaN,0
4,Imputación de Datos,Variables Climáticas y Entomológicas,Presencia de vacíos (especialmente en Nodos 15 y 16 por fallos de red),Interpolación lineal intra-nodo secuencial + Mediana espacio-temporal (Año-Mes),565


In [17]:
ruta_salida_excel = 'dataset_limpio_fase1.xlsx'
ruta_salida_bitacora = 'bitacora_decisiones_fase1.csv'

df.to_excel(ruta_salida_excel, index=False)

# Exportar bitácora en CSV
df_bitacora.to_csv(ruta_salida_bitacora, index=False)